In [ ]:
# ========== ИМПОРТЫ ==========

import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
import gc
import warnings
import os
import time
import yaml
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

# NLP библиотеки
import nltk
from nltk.tokenize import word_tokenize
import emoji

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Скачать токенизатор (один раз)
nltk.download('punkt')

warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split

# ROUGE метрика
from rouge_score import rouge_scorer

# Transformers для сравнения
from transformers import pipeline

In [ ]:
# ========== 1. КОНФИГУРАЦИЯ ==========
class Config:
    # Параметры данных
    min_word_freq = 2
    
    # Параметры модели LSTM
    lstm_embedding_dim = 128
    lstm_hidden_dim = 256
    lstm_num_layers = 2
    lstm_dropout = 0.3
    lstm_batch_size = 128
    lstm_epochs = 3
    lstm_lr = 0.001
    lstm_weight_decay = 1e-5
    
    # Параметры генерации
    max_seq_len = 80
    gen_temperature = 0.8
    gen_top_k = 50
    gen_top_p = 0.9
    
    # Другое
    random_seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()
print("="*60)
print("ФИНАЛЬНЫЙ СКРИПТ: LSTM vs DistilGPT2 (KAGGLE)")
print("="*60)
print(f"Используется устройство: {config.device}")
if config.device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
random.seed(config.random_seed)
np.random.seed(config.random_seed)
torch.manual_seed(config.random_seed)

In [ ]:
# ========== 2. ЗАГРУЗКА И ОЧИСТКА ДАННЫХ ==========
print("\n" + "="*60)
print("ЭТАП 1: ЗАГРУЗКА И ОЧИСТКА ДАННЫХ")
print("="*60)

# Скачиваем nltk данные
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# Загружаем файл
file_path = '/kaggle/input/datasets/viacheslavr/raw-dataset-txt/raw_dataset.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Убираем пустые строки
raw_texts = [line.strip() for line in lines if line.strip()]
print(f"Загружено текстов: {len(raw_texts)}")
print("Пример:", raw_texts[0])

def clean_text(text):
    # Удалить упоминания
    text = re.sub(r'@\w+', '', text)
    # Удалить ссылки
    text = re.sub(r'http\S+', '', text)
    # Удалить эмодзи
    text = emoji.replace_emoji(text, replace='')
    # Привести к нижнему регистру
    text = text.lower()
    # Удалить лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Очищаем
print("Очистка текстов...")
cleaned_texts = [clean_text(text) for text in tqdm(raw_texts)]

print("Оригинал:", raw_texts[0])
print("Очищенный:", cleaned_texts[0])

# Токенизация
print("\nТокенизация...")
tokenized_texts = [word_tokenize(text) for text in tqdm(cleaned_texts)]
print("Токены:", tokenized_texts[0][:10])

# Удаляем пустые тексты
non_empty = [i for i, tokens in enumerate(tokenized_texts) if len(tokens) > 0]
cleaned_texts = [cleaned_texts[i] for i in non_empty]
tokenized_texts = [tokenized_texts[i] for i in non_empty]
print(f"После фильтрации: {len(cleaned_texts)} текстов")

In [ ]:
# ========== 3. РАЗДЕЛЕНИЕ НА ТРЕЙН/ВАЛ/ТЕСТ (80/10/10) ==========
print("\n" + "="*60)
print("ЭТАП 1 (продолжение): РАЗДЕЛЕНИЕ ДАННЫХ")
print("="*60)

# Сначала разделяем на трейн (80%) и временные (20%)
train_texts, temp_texts, train_tokens, temp_tokens = train_test_split(
    cleaned_texts, tokenized_texts, 
    test_size=0.2, 
    random_state=config.random_seed
)

# Затем временные на вал (10%) и тест (10%)
val_texts, test_texts, val_tokens, test_tokens = train_test_split(
    temp_texts, temp_tokens, 
    test_size=0.5, 
    random_state=config.random_seed
)

print(f"Train: {len(train_texts)} ({len(train_texts)/len(cleaned_texts)*100:.1f}%)")
print(f"Validation: {len(val_texts)} ({len(val_texts)/len(cleaned_texts)*100:.1f}%)")
print(f"Test: {len(test_texts)} ({len(test_texts)/len(cleaned_texts)*100:.1f}%)")

In [ ]:
# ========== 4. ПОСТРОЕНИЕ СЛОВАРЯ ==========
print("\n" + "="*60)
print("ЭТАП 1 (продолжение): ПОСТРОЕНИЕ СЛОВАРЯ")
print("="*60)

def build_vocab(tokens_list, min_freq=2):
    word_counts = Counter()
    for tokens in tokens_list:
        word_counts.update(tokens)
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    
    return vocab, word_counts

vocab, word_counts = build_vocab(train_tokens, min_freq=config.min_word_freq)
print(f"Размер словаря: {len(vocab)}")
print(f"Токенов в обучающей выборке: {sum(word_counts.values()):,}")

# Обратный словарь
idx_to_word = {v: k for k, v in vocab.items()}

In [ ]:
# ========== 5. ПОДГОТОВКА DATASET И DATALOADER ==========
print("\n" + "="*60)
print("ЭТАП 1 (продолжение): ПОДГОТОВКА DATALOADER")
print("="*60)

def tokens_to_indices(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

class TextDataset(Dataset):
    def __init__(self, texts, tokens, vocab):
        self.texts = texts
        self.indices = [torch.tensor(tokens_to_indices(t, vocab), dtype=torch.long) 
                       for t in tokens]
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return {
            'text': self.texts[idx],
            'indices': self.indices[idx]
        }

def collate_batch(batch):
    texts = [item['text'] for item in batch]
    indices = [item['indices'] for item in batch]
    
    # Обрезаем длинные последовательности
    indices = [seq[:config.max_seq_len] for seq in indices]
    indices_padded = pad_sequence(indices, batch_first=True, padding_value=0)
    
    return {
        'texts': texts,
        'indices': indices_padded
    }

# Создаем датасеты
train_dataset = TextDataset(train_texts, train_tokens, vocab)
val_dataset = TextDataset(val_texts, val_tokens, vocab)
test_dataset = TextDataset(test_texts, test_tokens, vocab)

# Создаем DataLoader'ы
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.lstm_batch_size, 
    shuffle=True, 
    collate_fn=collate_batch,
    pin_memory=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.lstm_batch_size, 
    shuffle=False, 
    collate_fn=collate_batch,
    pin_memory=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=config.lstm_batch_size, 
    shuffle=False, 
    collate_fn=collate_batch,
    pin_memory=True,
    num_workers=2
)

print(f"Train батчей: {len(train_loader)}")
print(f"Val батчей: {len(val_loader)}")
print(f"Test батчей: {len(test_loader)}")

In [ ]:
# ========== 6. LSTM МОДЕЛЬ ==========
print("\n" + "="*60)
print("ЭТАП 2: РЕАЛИЗАЦИЯ LSTM МОДЕЛИ")
print("="*60)

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Эмбеддинги
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(dropout)
        
        # LSTM
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Выходные слои
        self.lstm_dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        # Инициализация весов
        self._init_weights()
    
    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
    
    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        emb = self.embedding_dropout(emb)
        
        lstm_out, hidden = self.lstm(emb, hidden)
        lstm_out = self.lstm_dropout(lstm_out)
        lstm_out = self.layer_norm(lstm_out)
        
        logits = self.fc(lstm_out)
        return logits, hidden
    
    def generate(self, start_tokens, max_length=20, temperature=0.8, top_k=50, top_p=0.9):
        self.eval()
        with torch.no_grad():
            current = start_tokens.unsqueeze(0).to(config.device)
            generated = current.clone()
            hidden = None
            
            for _ in range(max_length):
                logits, hidden = self.forward(current, hidden)
                logits = logits[0, -1, :] / temperature
                
                # Top-k фильтрация
                if top_k > 0:
                    top_k_values, top_k_indices = torch.topk(logits, top_k)
                    probs = torch.zeros_like(logits).fill_(-float('Inf'))
                    probs[top_k_indices] = top_k_values
                    logits = probs
                
                # Top-p фильтрация
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                    
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    
                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    logits[indices_to_remove] = -float('Inf')
                
                probs = torch.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, 1).unsqueeze(0)
                
                generated = torch.cat([generated, next_token], dim=1)
                current = next_token
            
            return generated.squeeze(0)

# Создаем модель
model = LSTMLanguageModel(
    vocab_size=len(vocab),
    embedding_dim=config.lstm_embedding_dim,
    hidden_dim=config.lstm_hidden_dim,
    num_layers=config.lstm_num_layers,
    dropout=config.lstm_dropout
).to(config.device)

print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ========== 7. ТРЕНИРОВКА LSTM ==========
print("\n" + "="*60)
print("ЭТАП 3: ТРЕНИРОВКА LSTM МОДЕЛИ")
print("="*60)

criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=config.lstm_lr, weight_decay=config.lstm_weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2)

train_losses = []
val_losses = []

best_val_loss = float('inf')

for epoch in range(config.lstm_epochs):
    # Тренировка
    model.train()
    total_train_loss = 0
    start_time = time.time()
    
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.lstm_epochs} [Train]')
    for batch_idx, batch in enumerate(progress):
        indices = batch['indices'].to(config.device, non_blocking=True)
        
        # Ограничиваем длину для экономии памяти
        if indices.size(1) > config.max_seq_len:
            indices = indices[:, :config.max_seq_len]
        
        input_seq = indices[:, :-1]
        target_seq = indices[:, 1:]
        
        optimizer.zero_grad()
        logits, _ = model(input_seq)
        loss = criterion(logits.reshape(-1, logits.size(-1)), target_seq.reshape(-1))
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        progress.set_postfix({'loss': f'{loss.item():.3f}'})
        
        # Очистка памяти
        del logits, loss, input_seq, target_seq, indices
        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()
    
    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Валидация
    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{config.lstm_epochs} [Val]'):
            indices = batch['indices'].to(config.device, non_blocking=True)
            
            if indices.size(1) > config.max_seq_len:
                indices = indices[:, :config.max_seq_len]
            
            input_seq = indices[:, :-1]
            target_seq = indices[:, 1:]
            
            logits, _ = model(input_seq)
            loss = criterion(logits.reshape(-1, logits.size(-1)), target_seq.reshape(-1))
            total_val_loss += loss.item()
            
            del logits, loss, input_seq, target_seq, indices
    
    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Сохраняем лучшую модель
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_lstm_model.pt')
        print(f"  ✓ Сохранена лучшая модель (val_loss: {avg_val_loss:.4f})")
    
    epoch_time = time.time() - start_time
    print(f"\nEpoch {epoch+1} Results (time: {epoch_time:.1f}s):")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Perplexity: {np.exp(avg_val_loss):.2f}")
    
    # Очистка после эпохи
    torch.cuda.empty_cache()
    gc.collect()

# Загружаем лучшую модель
model.load_state_dict(torch.load('best_lstm_model.pt', map_location=config.device))

In [ ]:
# ========== 8. ROUGE МЕТРИКИ ДЛЯ LSTM ==========
print("\n" + "="*60)
print("ЭТАП 3 (продолжение): ОЦЕНКА ROUGE ДЛЯ LSTM")
print("="*60)

def compute_rouge_lstm(model, dataloader, num_samples=100):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
    rouge1_scores = []
    rouge2_scores = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx * config.lstm_batch_size >= num_samples:
                break
                
            indices = batch['indices'].to(config.device)
            
            for i in range(len(indices)):
                text_indices = indices[i]
                text_indices = text_indices[text_indices != 0]
                
                if len(text_indices) < 10:
                    continue
                
                # Контекст (первые 70%) и референс (30%)
                split = int(len(text_indices) * 0.7)
                context = text_indices[:split]
                reference = text_indices[split:]
                
                if len(reference) < 2:
                    continue
                
                # Генерация
                generated = model.generate(context, max_length=len(reference), 
                                         temperature=0.8, top_k=50)
                generated_part = generated[split:]
                
                # Преобразуем в текст
                ref_text = ' '.join([idx_to_word.get(int(i), '') for i in reference.cpu()])
                gen_text = ' '.join([idx_to_word.get(int(i), '') for i in generated_part.cpu()])
                
                if ref_text and gen_text:
                    scores = scorer.score(ref_text, gen_text)
                    rouge1_scores.append(scores['rouge1'].fmeasure)
                    rouge2_scores.append(scores['rouge2'].fmeasure)
    
    return {
        'rouge1': np.mean(rouge1_scores) if rouge1_scores else 0,
        'rouge2': np.mean(rouge2_scores) if rouge2_scores else 0
    }

# Оценка на тесте
print("Оценка LSTM на тестовой выборке...")
lstm_rouge = compute_rouge_lstm(model, test_loader, num_samples=200)
print(f"LSTM ROUGE-1: {lstm_rouge['rouge1']:.4f}")
print(f"LSTM ROUGE-2: {lstm_rouge['rouge2']:.4f}")

In [ ]:
# ========== 9. DISTILGPT2 (ИСПРАВЛЕННАЯ ВЕРСИЯ) ==========
print("\n" + "="*60)
print("ЭТАП 4: ИСПОЛЬЗОВАНИЕ DISTILGPT2")
print("="*60)

class DistilGPT2Comparer:
    def __init__(self):
        print("Загрузка DistilGPT2...")
        # Убираем framework="pt" - pipeline сам определяет
        self.generator = pipeline(
            "text-generation", 
            model="distilgpt2",
            device=0 if torch.cuda.is_available() else -1
        )
    
    def generate(self, prompt, max_length=20):
        result = self.generator(
            prompt,
            max_length=max_length,
            do_sample=True,
            top_k=50,
            temperature=0.8,
            pad_token_id=50256,
            truncation=True
        )
        return result[0]['generated_text']

# Инициализируем GPT2
gpt2 = DistilGPT2Comparer()

In [ ]:
# ========== 10. ROUGE МЕТРИКИ ДЛЯ DISTILGPT2 ==========
print("\n" + "="*60)
print("ЭТАП 4 (продолжение): ОЦЕНКА ROUGE ДЛЯ DISTILGPT2")
print("="*60)

def compute_rouge_gpt2(gpt2, test_texts, num_samples=50):
    """Оценка ROUGE для DistilGPT2"""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
    rouge1_scores = []
    rouge2_scores = []
    
    # Берем случайные тексты
    samples = random.sample(test_texts, min(num_samples, len(test_texts)))
    
    for text in tqdm(samples, desc="Оценка GPT2"):
        words = text.split()
        if len(words) < 10:
            continue
        
        # Контекст (первые 70%) и референс (30%)
        split = int(len(words) * 0.7)
        context = ' '.join(words[:split])
        reference = ' '.join(words[split:])
        
        if len(reference.split()) < 2:
            continue
        
        # Генерация через GPT2
        try:
            generated = gpt2.generate(context, max_length=len(words))
            generated_words = generated.split()
            
            if len(generated_words) > split:
                generated_part = ' '.join(generated_words[split:split + len(reference.split())])
                
                # Считаем ROUGE
                scores = scorer.score(reference, generated_part)
                rouge1_scores.append(scores['rouge1'].fmeasure)
                rouge2_scores.append(scores['rouge2'].fmeasure)
        except Exception as e:
            continue
    
    return {
        'rouge1': np.mean(rouge1_scores) if rouge1_scores else 0,
        'rouge2': np.mean(rouge2_scores) if rouge2_scores else 0
    }

# Оцениваем GPT2
print("Оценка DistilGPT2 на тестовой выборке...")
gpt2_rouge = compute_rouge_gpt2(gpt2, test_texts[:500], num_samples=50)

print(f"\nGPT2 ROUGE-1: {gpt2_rouge['rouge1']:.4f}")
print(f"GPT2 ROUGE-2: {gpt2_rouge['rouge2']:.4f}")

In [ ]:

# ========== 11. СРАВНЕНИЕ ГЕНЕРАЦИИ ==========
print("\n" + "="*60)
print("ЭТАП 4 (продолжение): СРАВНЕНИЕ ГЕНЕРАЦИИ")
print("="*60)

test_prompts = [
    "i love",
    "today is",
    "i don't know",
    "the weather",
    "i think that",
    "good morning"
]

print("\nСРАВНЕНИЕ ПРИМЕРОВ ГЕНЕРАЦИИ:")
print("-" * 70)
print(f"{'Prompt':<20} {'LSTM':<30} {'GPT2':<30}")
print("-" * 70)

for prompt in test_prompts:
    # LSTM
    tokens = prompt.lower().split()
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    indices = torch.tensor(indices).to(config.device)
    
    with torch.no_grad():
        lstm_gen = model.generate(indices, max_length=8, temperature=0.8)
    
    lstm_words = [idx_to_word.get(int(i), '<UNK>') for i in lstm_gen.cpu()]
    lstm_text = ' '.join(lstm_words)
    
    # GPT2
    gpt2_text = gpt2.generate(prompt, max_length=len(tokens) + 8)
    
    print(f"{prompt:<20} {lstm_text[:30]:<30} {gpt2_text[:30]:<30}")

In [ ]:
# ========== 12. ВИЗУАЛИЗАЦИЯ ==========
print("\n" + "="*60)
print("ЭТАП 5: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*60)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.grid(True)

plt.subplot(1, 3, 2)
perplexities = [np.exp(l) for l in val_losses]
plt.plot(perplexities, label='Perplexity', marker='o', color='red')
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.title('Validation Perplexity')
plt.grid(True)

plt.subplot(1, 3, 3)
models = ['LSTM', 'DistilGPT2']
rouge1_scores = [lstm_rouge['rouge1'], gpt2_rouge['rouge1']]
rouge2_scores = [lstm_rouge['rouge2'], gpt2_rouge['rouge2']]
x = np.arange(len(models))
width = 0.35
plt.bar(x - width/2, rouge1_scores, width, label='ROUGE-1', color='steelblue')
plt.bar(x + width/2, rouge2_scores, width, label='ROUGE-2', color='coral')
plt.xlabel('Model')
plt.ylabel('Score')
plt.title('ROUGE Scores Comparison')
plt.xticks(x, models)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150)
plt.show()


In [ ]:
# ========== 13. СОХРАНЕНИЕ КОНФИГОВ И МОДЕЛЕЙ ==========
print("\n" + "="*60)
print("СОХРАНЕНИЕ КОНФИГОВ И МОДЕЛЕЙ")
print("="*60)

# Создаем папки
os.makedirs('configs', exist_ok=True)
os.makedirs('models', exist_ok=True)

# ===== 1. СОХРАНЯЕМ КОНФИГИ (YAML) =====
print("\n📁 Сохраняем конфиги в configs/...")

# Основной конфиг модели
model_config = {
    'model': {
        'type': 'LSTM',
        'vocab_size': len(vocab),
        'embedding_dim': config.lstm_embedding_dim,
        'hidden_dim': config.lstm_hidden_dim,
        'num_layers': config.lstm_num_layers,
        'dropout': config.lstm_dropout
    },
    'training': {
        'batch_size': config.lstm_batch_size,
        'epochs': config.lstm_epochs,
        'learning_rate': config.lstm_lr,
        'weight_decay': config.lstm_weight_decay,
        'max_seq_len': config.max_seq_len
    },
    'generation': {
        'temperature': config.gen_temperature,
        'top_k': config.gen_top_k,
        'top_p': config.gen_top_p
    },
    'data': {
        'min_word_freq': config.min_word_freq,
        'vocab_size': len(vocab),
        'train_samples': len(train_texts),
        'val_samples': len(val_texts),
        'test_samples': len(test_texts)
    }
}

with open('configs/model_config.yaml', 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)
print("  ✅ configs/model_config.yaml")

# Конфиг с метриками
metrics_config = {
    'lstm': {
        'rouge1': float(lstm_rouge['rouge1']),
        'rouge2': float(lstm_rouge['rouge2']),
        'best_val_loss': float(best_val_loss),
        'final_perplexity': float(np.exp(val_losses[-1])),
        'train_losses': [float(x) for x in train_losses],
        'val_losses': [float(x) for x in val_losses]
    },
    'gpt2': {
        'rouge1': float(gpt2_rouge['rouge1']),
        'rouge2': float(gpt2_rouge['rouge2'])
    },
    'comparison': {
        'rouge1_diff': float(lstm_rouge['rouge1'] - gpt2_rouge['rouge1']),
        'rouge2_diff': float(lstm_rouge['rouge2'] - gpt2_rouge['rouge2']),
        'better_model': 'LSTM' if lstm_rouge['rouge1'] > gpt2_rouge['rouge1'] else 'GPT2'
    }
}

with open('configs/metrics_config.yaml', 'w') as f:
    yaml.dump(metrics_config, f, default_flow_style=False)
print("  ✅ configs/metrics_config.yaml")

# ===== 2. СОХРАНЯЕМ МОДЕЛИ =====
print("\n📁 Сохраняем модели в models/...")

# Копируем лучшую модель в папку models
import shutil
shutil.copy2('best_lstm_model.pt', 'models/best_lstm_model.pt')
print("  ✅ models/best_lstm_model.pt")

# Сохраняем также словарь для модели
torch.save(vocab, 'models/vocab.pt')
print("  ✅ models/vocab.pt")

# Сохраняем конфиг устройства
device_config = {
    'device': str(config.device),
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
}

with open('configs/device_config.yaml', 'w') as f:
    yaml.dump(device_config, f, default_flow_style=False)
print("  ✅ configs/device_config.yaml")

print("\n" + "="*60)
print("🎉 ГОТОВО! Сохранены конфиги и модели")
print("="*60)
print("\n📂 Созданные папки и файлы:")
print("  📁 configs/")
print("     ├── model_config.yaml    - конфиг модели")
print("     ├── metrics_config.yaml  - метрики")
print("     └── device_config.yaml   - информация об устройстве")
print("  📁 models/")
print("     ├── best_lstm_model.pt   - веса модели")
print("     └── vocab.pt             - словарь")
print("\n✅ Скрипт выполнен успешно!")